In [2]:
"""
Stage 5 - RAG-Augmented Incident Report Generation
"""

import json
import time
import requests
import pandas as pd
import numpy as np
import chromadb
from pathlib import Path
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# ── Paths ─────────────────────────────────────────────────────────────────────
RESULTS_DIR   = Path("../results")
STAGE4_CSV    = RESULTS_DIR / "stage4/all_results.csv"
COMMUNITY_CSV = RESULTS_DIR / "community_assignments.csv"
STIX_FILE     = Path("../data/attck/enterprise-attack.json")
CHROMADB_DIR  = Path("../chromadb")
OUTPUT_DIR    = RESULTS_DIR / "stage5"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

#  Parameters 
OLLAMA_URL  = "http://localhost:11434/api/generate"
MODEL       = "qwen2.5:3b"
EMBED_MODEL = "all-MiniLM-L6-v2"
TOP_K       = 3        
RANDOM_SEED = 1234
TEMPERATURE = 0.0
TIMEOUT     = 300      
COLLECTION  = "attck_procedures"

#  Step 1: Load ATT&CK STIX and extract procedure descriptions 
def load_attck_procedures():
    """
    Extract procedure descriptions from MITRE ATT&CK STIX bundle.
    """
    print("Loading ATT&CK procedure descriptions from STIX...")
    with open(STIX_FILE, "r", encoding="utf-8") as f:
        bundle = json.load(f)

    procedures = []
    techniques = {}

    # First pass — build technique lookup
    for obj in bundle.get("objects", []):
        if obj.get("type") != "attack-pattern":
            continue
        if obj.get("revoked", False) or obj.get("deprecated", False):
            continue
        tech_id = None
        for ref in obj.get("external_references", []):
            if ref.get("source_name") == "mitre-attack":
                tech_id = ref.get("external_id")
                break
        if not tech_id:
            continue
        tactics = []
        for phase in obj.get("kill_chain_phases", []):
            if phase.get("kill_chain_name") == "mitre-attack":
                tactics.append(
                    phase["phase_name"].replace("-", " ").title()
                )
        techniques[obj["id"]] = {
            "technique_id":   tech_id,
            "technique_name": obj.get("name", ""),
            "tactic":         tactics[0] if tactics else "Unknown",
            "description":    obj.get("description", "")[:300],
        }

    # Second pass — extract procedure examples from relationships
    for obj in bundle.get("objects", []):
        if obj.get("type") != "relationship":
            continue
        if obj.get("relationship_type") != "uses":
            continue
        target_id = obj.get("target_ref", "")
        if target_id not in techniques:
            continue
        procedure_text = obj.get("description", "").strip()
        if not procedure_text or len(procedure_text) < 50:
            continue
        tech = techniques[target_id]
        procedures.append({
            "id": obj["id"],
            "technique_id":   tech["technique_id"],
            "technique_name": tech["technique_name"],
            "tactic":         tech["tactic"],
            "procedure_text": procedure_text[:250],
            "full_text": (
                f"Technique: {tech['technique_id']} "
                f"{tech['technique_name']} "
                f"[{tech['tactic']}]. "
                f"Procedure: {procedure_text[:250]}"
            )
        })

    # Also add technique descriptions themselves
    for tech_obj_id, tech in techniques.items():
        if tech["description"]:
            procedures.append({
                "id":             f"desc_{tech['technique_id']}",
                "technique_id":   tech["technique_id"],
                "technique_name": tech["technique_name"],
                "tactic":         tech["tactic"],
                "procedure_text": tech["description"],
                "full_text": (
                    f"Technique: {tech['technique_id']} "
                    f"{tech['technique_name']} "
                    f"[{tech['tactic']}]. "
                    f"Description: {tech['description']}"
                )
            })

    print(f"  Extracted {len(procedures):,} procedure/description entries")
    return procedures

# Build ChromaDB knowledge base 
def build_chromadb(procedures):
    """
    Embed all ATT&CK procedure descriptions and store in ChromaDB.
    Uses all-MiniLM-L6-v2 for consistency with Stage 2 embeddings.
    """
    print(f"\nBuilding ChromaDB knowledge base...")
    client = chromadb.PersistentClient(path=str(CHROMADB_DIR))

    # Delete existing collection if present
    try:
        client.delete_collection(COLLECTION)
        print(f"  Deleted existing collection")
    except Exception:
        pass

    collection = client.create_collection(
        name     = COLLECTION,
        metadata = {"hnsw:space": "cosine"}
    )

    print(f"  Loading embedding model: {EMBED_MODEL}")
    embedder = SentenceTransformer(EMBED_MODEL)

    texts     = [p["full_text"] for p in procedures]
    ids       = [p["id"][:64]   for p in procedures]
    metadatas = [{
        "technique_id":   p["technique_id"],
        "technique_name": p["technique_name"],
        "tactic":         p["tactic"],
    } for p in procedures]

    print(f"  Embedding {len(texts):,} entries...")
    batch_size     = 256
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="  Embedding"):
        batch = texts[i:i + batch_size]
        embs  = embedder.encode(
            batch,
            normalize_embeddings=True,
            show_progress_bar=False
        )
        all_embeddings.extend(embs.tolist())

    print(f"  Adding to ChromaDB...")
    for i in tqdm(range(0, len(ids), 500), desc="  Inserting"):
        end = min(i + 500, len(ids))
        collection.add(
            ids        = ids[i:end],
            embeddings = all_embeddings[i:end],
            documents  = texts[i:end],
            metadatas  = metadatas[i:end],
        )

    print(f"  ChromaDB built: {collection.count():,} entries")
    return client, collection, embedder

# Retrieve relevant ATT&CK context 
def retrieve_attck_context(query, collection, embedder, technique_id=None):
    """
    Retrieve top-K most relevant ATT&CK procedure descriptions.
    Prioritises results matching the predicted technique ID.
    """
    query_emb = embedder.encode(
        query, normalize_embeddings=True
    ).tolist()

    results = collection.query(
        query_embeddings = [query_emb],
        n_results        = TOP_K * 2,
        include          = ["documents", "metadatas", "distances"]
    )

    docs      = results["documents"][0]
    metas     = results["metadatas"][0]
    distances = results["distances"][0]

    ranked = list(zip(docs, metas, distances))
    if technique_id:
        exact  = [(d, m, s) for d, m, s in ranked
                  if m.get("technique_id") == technique_id]
        others = [(d, m, s) for d, m, s in ranked
                  if m.get("technique_id") != technique_id]
        ranked = (exact + others)[:TOP_K]
    else:
        ranked = ranked[:TOP_K]

    context_texts = [d for d, m, s in ranked]
    context_metas = [m for d, m, s in ranked]
    return context_texts, context_metas

# Build report prompts 
def build_rag_prompt(community_desc, technique_id,
                     technique_name, tactic, context_texts):
    # Truncate each reference to keep total prompt manageable
    context_block = "\n\n".join([
        f"[Ref {i+1}]: {text[:200]}"
        for i, text in enumerate(context_texts[:3])
    ])
    return (
        "You are a senior cybersecurity analyst writing a structured "
        "incident report.\n\n"
        f"DETECTED INCIDENT:\n{community_desc}\n\n"
        f"ATT&CK TECHNIQUE: {technique_id} — {technique_name} [{tactic}]\n\n"
        f"RETRIEVED ATT&CK CONTEXT:\n{context_block}\n\n"
        "Using the ATT&CK context as your knowledge source, write a "
        "structured incident report. Ground every technical claim in "
        "the retrieved context.\n\n"
        "Return JSON with exactly these fields:\n"
        "- incident_summary: 2 sentences describing what happened\n"
        "- technique_id: ATT&CK technique ID\n"
        "- technique_name: ATT&CK technique name\n"
        "- tactic: ATT&CK tactic\n"
        "- severity: Critical, High, Medium, or Low\n"
        "- evidence: list of 3 observable indicators\n"
        "- containment_steps: list of 3 immediate actions\n"
        "- investigation_steps: list of 3 investigation actions\n"
        "- attck_grounded: true\n\n"
        "JSON only. No other text."
    )

def build_no_rag_prompt(community_desc, technique_id,
                        technique_name, tactic):
    return (
        "You are a senior cybersecurity analyst writing a structured "
        "incident report.\n\n"
        f"DETECTED INCIDENT:\n{community_desc}\n\n"
        f"ATT&CK TECHNIQUE: {technique_id} — {technique_name} [{tactic}]\n\n"
        "Write a structured incident report based on your knowledge.\n\n"
        "Return JSON with exactly these fields:\n"
        "- incident_summary: 2 sentences describing what happened\n"
        "- technique_id: ATT&CK technique ID\n"
        "- technique_name: ATT&CK technique name\n"
        "- tactic: ATT&CK tactic\n"
        "- severity: Critical, High, Medium, or Low\n"
        "- evidence: list of 3 observable indicators\n"
        "- containment_steps: list of 3 immediate actions\n"
        "- investigation_steps: list of 3 investigation actions\n"
        "- attck_grounded: false\n\n"
        "JSON only. No other text."
    )

# Call Ollama 
def call_ollama(prompt):
    payload = {
        "model":  MODEL,
        "prompt": prompt,
        "stream": False,
        "format": "json",
        "options": {
            "temperature": TEMPERATURE,
            "seed":        RANDOM_SEED,
            "num_predict": 500,
            "think":       False,
        }
    }
    try:
        resp = requests.post(
            OLLAMA_URL, json=payload, timeout=TIMEOUT
        )
        resp.raise_for_status()
        raw = resp.json().get("response", "").strip()
        raw = raw.replace("```json", "").replace("```", "").strip()
        if not raw:
            return None
        return json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"    JSON error: {e}")
        return None
    except requests.exceptions.Timeout:
        print(f"    Timeout after {TIMEOUT}s")
        return None
    except Exception as e:
        print(f"    Error: {e}")
        return None

# Factual consistency scoring 
def score_factual_consistency(report, context_texts, technique_id):
    """
    Measure factual consistency of generated report against
    retrieved ATT&CK context.

    Metric: weighted score across three dimensions:
      - Technique grounding (30%): technique ID present in context
      - Evidence support (40%): evidence items overlap with context
      - Containment support (30%): containment steps overlap with context

    This operationalises the RQ3 evaluation framework.
    """
    if report is None:
        return {
            "consistency_score":  0.0,
            "technique_grounded": False,
            "evidence_score":     0.0,
            "containment_score":  0.0,
            "has_evidence":       False,
            "has_containment":    False,
            "has_investigation":  False,
            "severity":           "",
            "parse_failed":       True,
        }

    context_combined = " ".join(context_texts).lower()

    # Technique ID in context
    tech_in_context = technique_id.lower() in context_combined

    # Evidence support
    stop = {
        "the","a","an","is","in","of","to","and","or",
        "with","for","on","at","by","from","this","that",
        "are","was","were","be","been","has","have","had"
    }
    evidence = report.get("evidence", [])
    if isinstance(evidence, list) and len(evidence) > 0:
        ev_scores = []
        for item in evidence[:3]:
            words = set(str(item).lower().split()) - stop
            if not words:
                ev_scores.append(0.0)
                continue
            overlap = sum(1 for w in words if w in context_combined)
            ev_scores.append(overlap / len(words))
        evidence_score = float(np.mean(ev_scores))
    else:
        evidence_score = 0.0

    # Containment support
    containment = report.get("containment_steps", [])
    if isinstance(containment, list) and len(containment) > 0:
        all_words = set(
            " ".join([str(c) for c in containment]).lower().split()
        ) - stop
        if all_words:
            overlap = sum(1 for w in all_words if w in context_combined)
            containment_score = min(overlap / len(all_words), 1.0)
        else:
            containment_score = 0.0
    else:
        containment_score = 0.0

    # Weighted overall score
    consistency = (
        0.30 * float(tech_in_context) +
        0.40 * evidence_score +
        0.30 * containment_score
    )

    return {
        "consistency_score":  round(consistency, 4),
        "technique_grounded": tech_in_context,
        "evidence_score":     round(evidence_score, 4),
        "containment_score":  round(containment_score, 4),
        "has_evidence":       isinstance(evidence, list) and len(evidence) > 0,
        "has_containment":    isinstance(containment, list) and len(containment) > 0,
        "has_investigation":  isinstance(
            report.get("investigation_steps", []), list
        ) and len(report.get("investigation_steps", [])) > 0,
        "severity":           report.get("severity", ""),
        "parse_failed":       False,
    }

# Community description builder 
def build_community_description(comm_df):
    n = len(comm_df)

    def safe_mean(col):
        if col not in comm_df.columns:
            return 0.0
        vals = pd.to_numeric(comm_df[col], errors="coerce").dropna()
        return round(float(vals.mean()), 1) if len(vals) > 0 else 0.0

    label_counts   = comm_df["Label"].value_counts()
    dominant_label = label_counts.index[0]
    label_pct      = label_counts.iloc[0] / n * 100

    if "Destination Port" in comm_df.columns:
        top_ports = comm_df["Destination Port"].value_counts().head(2)
        port_str  = ", ".join(
            [f"port {p} ({c} flows)" for p, c in top_ports.items()]
        )
    else:
        port_str = "unknown"

    fwd  = safe_mean("Total Fwd Packets")
    bwd  = safe_mean("Total Backward Packets")
    bps  = safe_mean("Flow Bytes/s")
    dur  = safe_mean("Flow Duration")
    pmean= safe_mean("Packet Length Mean")
    syn  = safe_mean("SYN Flag Count")
    rst  = safe_mean("RST Flag Count")
    ack  = safe_mean("ACK Flag Count")

    flags = []
    if syn  > 0.1: flags.append(f"SYN(avg={syn:.1f})")
    if rst  > 0.1: flags.append(f"RST(avg={rst:.1f})")
    if ack  > 0.1: flags.append(f"ACK(avg={ack:.1f})")
    flag_str = ", ".join(flags) if flags else "none prominent"

    texts      = comm_df["alert_text"].dropna().head(2).tolist()
    text_block = "\n".join([f"  - {t[:100]}" for t in texts])

    return (
        f"Size: {n} alerts | "
        f"Label: {dominant_label} ({label_pct:.0f}%)\n"
        f"Ports: {port_str}\n"
        f"Duration: {dur:.0f}us | "
        f"Fwd pkts: {fwd} | Bwd pkts: {bwd}\n"
        f"Packet length mean: {pmean} bytes | "
        f"Flow rate: {bps:.0f} bytes/s\n"
        f"TCP flags: {flag_str}\n"
        f"Samples:\n{text_block}"
    )

# Main pipeline 
def main():
        # Load correctly classified communities from Stage 4
    print("\nLoading Stage 4 classification results...")
    stage4_df = pd.read_csv(STAGE4_CSV)
    correct   = stage4_df[
        (stage4_df["condition"] == "zero_shot") &
        (stage4_df["exact_match"] == True)
    ].copy()
    print(f"  Using {len(correct)} correctly classified communities")

    # Load community alerts
    comm_df_full = pd.read_csv(COMMUNITY_CSV, low_memory=False)

    # Build knowledge base
    procedures                   = load_attck_procedures()
    client, collection, embedder = build_chromadb(procedures)

    all_results = []
    print(f"\nGenerating reports ({len(correct)} communities × 2 conditions)...")

    for _, row in tqdm(correct.iterrows(),
                       total=len(correct),
                       desc="  Communities"):
        comm_id      = row["community_id"]
        technique_id = row["ground_truth_id"]
        tactic       = row["ground_truth_tactic"].title()
        tech_name    = str(row.get("predicted_name", technique_id))

        comm_alerts = comm_df_full[
            comm_df_full["community_id"] == comm_id
        ].copy()
        if len(comm_alerts) == 0:
            continue

        community_desc = build_community_description(comm_alerts)

        # Build retrieval query
        dominant_label = (
            comm_alerts["Label"].mode()[0]
            if len(comm_alerts) > 0 else ""
        )
        query = (
            f"{technique_id} {tech_name} {tactic} {dominant_label}"
        )

        context_texts, context_metas = retrieve_attck_context(
            query, collection, embedder, technique_id
        )

        for condition in ["rag", "no_rag"]:
            if condition == "rag":
                prompt = build_rag_prompt(
                    community_desc, technique_id,
                    tech_name, tactic, context_texts
                )
            else:
                prompt = build_no_rag_prompt(
                    community_desc, technique_id,
                    tech_name, tactic
                )

            start   = time.time()
            report  = call_ollama(prompt)
            elapsed = time.time() - start

            scores = score_factual_consistency(
                report,
                context_texts if condition == "rag" else [],
                technique_id
            )

            result = {
                "community_id":   comm_id,
                "community_size": len(comm_alerts),
                "technique_id":   technique_id,
                "tactic":         tactic,
                "condition":      condition,
                "latency_s":      round(elapsed, 2),
                "n_retrieved":    len(context_texts) if condition == "rag" else 0,
            }
            result.update(scores)
            result["incident_summary"] = (
                str(report.get("incident_summary", ""))[:300]
                if report and not scores["parse_failed"] else ""
            )
            all_results.append(result)

    # Save
    results_df  = pd.DataFrame(all_results)
    output_path = OUTPUT_DIR / "stage5_results.csv"
    results_df.to_csv(output_path, index=False)

    # Summary 
    print(f"EVALUATION SUMMARY")
    print(f"{'='*65}")

    for condition in ["rag", "no_rag"]:
        grp = results_df[
            (results_df["condition"] == condition) &
            (~results_df["parse_failed"])
        ]
        if len(grp) == 0:
            continue
        fails = results_df[
            results_df["condition"] == condition
        ]["parse_failed"].sum()
        print(f"\n  Condition : {condition.upper()}")
        print(f"    Reports generated      : {len(grp)}")
        print(f"    Parse failures         : {fails}")
        print(f"    Mean consistency score : "
              f"{grp['consistency_score'].mean():.4f}")
        print(f"    Mean evidence score    : "
              f"{grp['evidence_score'].mean():.4f}")
        print(f"    Mean containment score : "
              f"{grp['containment_score'].mean():.4f}")
        print(f"    Mean latency           : "
              f"{grp['latency_s'].mean():.1f}s")

    rag_scores   = results_df[
        (results_df["condition"] == "rag") &
        (~results_df["parse_failed"])
    ]["consistency_score"]
    norag_scores = results_df[
        (results_df["condition"] == "no_rag") &
        (~results_df["parse_failed"])
    ]["consistency_score"]

    if len(rag_scores) > 0 and len(norag_scores) > 0:
        improvement = rag_scores.mean() - norag_scores.mean()
        direction   = (
            "SUPPORTS" if improvement > 0 else "DOES NOT SUPPORT"
        )
        print(f"\n  RQ3 RESULT:")
        print(f"    RAG mean consistency   : {rag_scores.mean():.4f}")
        print(f"    No-RAG mean consistency: {norag_scores.mean():.4f}")
        print(f"    Improvement from RAG   : {improvement:+.4f}")
        print(
            f"    Finding: RAG {direction} the hypothesis that "
            f"retrieval reduces unsupported claims."
        )

    # Sample report
    sample = results_df[
        (results_df["condition"] == "rag") &
        (~results_df["parse_failed"])
    ].head(1)
    if len(sample) > 0:
        s = sample.iloc[0]
        print(f"\n  Sample RAG report (Community {int(s['community_id'])}):")
        print(f"    Technique   : {s['technique_id']}")
        print(f"    Severity    : {s['severity']}")
        print(f"    Consistency : {s['consistency_score']:.4f}")
        print(f"    Summary     : {s['incident_summary'][:200]}")

    print(f"\n  Results saved to: {output_path}")

if __name__ == "__main__":
    main()



Loading Stage 4 classification results...
  Using 13 correctly classified communities
Loading ATT&CK procedure descriptions from STIX...
  Extracted 16,720 procedure/description entries

Building ChromaDB knowledge base...
  Deleted existing collection
  Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Embedding 16,720 entries...


  Embedding: 100%|██████████| 66/66 [00:16<00:00,  4.07it/s]


  Adding to ChromaDB...


  Inserting: 100%|██████████| 34/34 [00:16<00:00,  2.08it/s]


  ChromaDB built: 16,720 entries

Generating reports (13 communities × 2 conditions)...


  Communities: 100%|██████████| 13/13 [11:11<00:00, 51.67s/it]

EVALUATION SUMMARY

  Condition : RAG
    Reports generated      : 13
    Parse failures         : 0
    Mean consistency score : 0.4420
    Mean evidence score    : 0.2489
    Mean containment score : 0.1413
    Mean latency           : 26.4s

  Condition : NO_RAG
    Reports generated      : 13
    Parse failures         : 0
    Mean consistency score : 0.0000
    Mean evidence score    : 0.0000
    Mean containment score : 0.0000
    Mean latency           : 25.3s

  RQ3 RESULT:
    RAG mean consistency   : 0.4420
    No-RAG mean consistency: 0.0000
    Improvement from RAG   : +0.4420
    Finding: RAG SUPPORTS the hypothesis that retrieval reduces unsupported claims.

  Sample RAG report (Community 32):
    Technique   : T1046
    Severity    : High
    Consistency : 0.3867
    Summary     : A PortScan incident was detected targeting ports 445 and 4445, with a duration of approximately 1 hour and 27 minutes. The attacker used network service discovery techniques to identify open se